# Pipeline B: Generative / RAG QA on SQuAD v2.0
**LLM**: Llama 3 (via Ollama — runs locally)  
**Dataset**: SQuAD v2.0  
**Evaluation**: Faithfulness (LLM-as-a-Judge) + Answer Relevance  
**Key feature**: Explicitly outputs `"Unanswerable"` when the context does not support an answer

## 1. Prerequisites

### 1.1 Install Ollama
```bash
# macOS / Linux — run once in terminal
curl -fsSL https://ollama.com/install.sh | sh
ollama pull llama3          # ~4.7 GB — the answering model
ollama pull llama3          # same model doubles as judge
```
Start the server (if not already running):
```bash
ollama serve &
```

In [ ]:
import subprocess, sys
for p in ['requests', 'datasets', 'numpy', 'matplotlib', 'tqdm', 'sentence-transformers']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', p])
print('Packages ready.')

In [ ]:
import re, json, time, string, warnings
import numpy as np
import matplotlib.pyplot as plt
import requests
from tqdm.auto import tqdm
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, util

warnings.filterwarnings('ignore')

OLLAMA_BASE  = 'http://localhost:11434'
ANSWER_MODEL = 'llama3'   # answering model
JUDGE_MODEL  = 'llama3'   # evaluator (can be a larger model if available)

# Verify Ollama is running
try:
    r = requests.get(f'{OLLAMA_BASE}/api/tags', timeout=5)
    available = [m['name'] for m in r.json().get('models', [])]
    print('Ollama is running. Available models:', available)
except Exception as e:
    print(f'ERROR: Could not reach Ollama at {OLLAMA_BASE}. Start with: ollama serve')
    print(f'Details: {e}')

## 2. Load SQuAD v2.0 Evaluation Subset

In [ ]:
dataset = load_dataset('squad_v2')
val     = dataset['validation']

answerable   = [ex for ex in val if len(ex['answers']['text']) > 0]
unanswerable = [ex for ex in val if len(ex['answers']['text']) == 0]

np.random.seed(42)
# Use 50 of each class to keep total LLM calls manageable
idx_a = np.random.choice(len(answerable),   50, replace=False).tolist()
idx_u = np.random.choice(len(unanswerable), 50, replace=False).tolist()

eval_samples = (
    [answerable[i]   for i in idx_a] +
    [unanswerable[i] for i in idx_u]
)
np.random.shuffle(eval_samples)

print(f'Evaluation set: {len(eval_samples)} samples  '
      f'({sum(1 for e in eval_samples if e["answers"]["text"])} answerable, '
      f'{sum(1 for e in eval_samples if not e["answers"]["text"])} unanswerable)')

## 3. RAG Pipeline Architecture

```
Question + Context
        │
        ▼
  ┌─────────────────────────────────────┐
  │  Prompt Builder                     │
  │  (system + few-shot + instructions) │
  └─────────────────────────────────────┘
        │
        ▼
  ┌─────────────────────────────────────┐
  │  Llama 3  (via Ollama REST API)     │
  └─────────────────────────────────────┘
        │
        ▼
  Generated Answer  OR  "Unanswerable"
        │
        ▼
  ┌───────────────────────────────────────────┐
  │  Evaluator                                │
  │  ├─ Faithfulness  (LLM-as-a-Judge)        │
  │  └─ Answer Relevance  (semantic sim.)     │
  └───────────────────────────────────────────┘
```

## 4. Ollama Helper & Prompt Builder

In [ ]:
def ollama_generate(prompt: str, model: str = ANSWER_MODEL,
                    temperature: float = 0.0,
                    max_tokens: int = 256) -> str:
    """
    Calls the Ollama REST API and returns the generated text.
    temperature=0 gives deterministic outputs for reproducibility.
    """
    payload = {
        'model'  : model,
        'prompt' : prompt,
        'stream' : False,
        'options': {
            'temperature' : temperature,
            'num_predict' : max_tokens,
            'top_p'       : 0.9
        }
    }
    response = requests.post(f'{OLLAMA_BASE}/api/generate',
                             json=payload, timeout=120)
    response.raise_for_status()
    return response.json()['response'].strip()


# QA system prompt
QA_SYSTEM = (
    'You are a precise question-answering assistant. '
    'Answer ONLY using information from the provided context. '
    'If the context does not contain enough information to answer the question, '
    'respond with exactly one word: Unanswerable. '
    'Keep answers concise — one or two sentences maximum.'
)


def build_qa_prompt(question: str, context: str) -> str:
    return (
        f'### System\n{QA_SYSTEM}\n\n'
        f'### Context\n{context.strip()}\n\n'
        f'### Question\n{question.strip()}\n\n'
        f'### Answer\n'
    )


print('Ollama helper defined.')

## 5. Run RAG Inference

In [ ]:
rag_results = []

for sample in tqdm(eval_samples, desc='RAG QA inference'):
    question = sample['question']
    context  = sample['context']
    golds    = sample['answers']['text']

    prompt  = build_qa_prompt(question, context)
    answer  = ollama_generate(prompt, model=ANSWER_MODEL)

    is_unanswerable_pred = 'unanswerable' in answer.lower().split()[:3]

    rag_results.append({
        'id'                    : sample['id'],
        'question'              : question,
        'context'               : context,
        'gold'                  : golds,
        'generated_answer'      : answer,
        'predicted_unanswerable': is_unanswerable_pred,
        'is_truly_unanswerable' : len(golds) == 0
    })

print(f'Inference complete: {len(rag_results)} samples.')

In [ ]:
# Display a few outputs
for ex in rag_results[:5]:
    label = 'UNANSWERABLE' if ex['is_truly_unanswerable'] else 'ANSWERABLE'
    print(f'[{label}]')
    print(f'  Q : {ex["question"]}')
    print(f'  Gold: {ex["gold"] if ex["gold"] else "<none>"}')
    print(f'  Gen : {ex["generated_answer"][:150]}')
    print(f'  Pred unanswerable: {ex["predicted_unanswerable"]}')
    print()

## 6. Unanswerability Performance

In [ ]:
truly_unans = [r for r in rag_results if r['is_truly_unanswerable']]
truly_ans   = [r for r in rag_results if not r['is_truly_unanswerable']]

true_pos  = sum(1 for r in truly_unans if r['predicted_unanswerable'])
false_neg = len(truly_unans) - true_pos
false_pos = sum(1 for r in truly_ans if r['predicted_unanswerable'])
true_neg  = len(truly_ans) - false_pos

precision = true_pos / (true_pos + false_pos + 1e-9)
recall    = true_pos / (true_pos + false_neg + 1e-9)
f1        = 2 * precision * recall / (precision + recall + 1e-9)

print('Unanswerability Detection')
print(f'  True Positives  (correctly flagged unanswerable) : {true_pos}')
print(f'  False Negatives (missed unanswerable)            : {false_neg}')
print(f'  False Positives (hallucinated "unanswerable")    : {false_pos}')
print(f'  Precision : {precision:.4f}')
print(f'  Recall    : {recall:.4f}')
print(f'  F1        : {f1:.4f}')

## 7. Evaluation Metric 1 — Faithfulness (LLM-as-a-Judge)

### Mathematical Definition

Faithfulness measures whether **every atomic claim** in the generated answer is
entailed by the provided context:

$$\text{Faithfulness}(A, C) = \frac{|\text{Supported Claims}(A, C)|}{|\text{All Claims}(A)|}$$

A claim is *supported* if a human (or LLM judge) can verify it directly from $C$
without external knowledge. A score of 1.0 means the answer is **grounded**;
a score below 1.0 indicates **hallucination**.

### Implementation
We use an LLM (Llama 3) as the judge with a structured decomposition prompt:
1. The judge extracts atomic claims from the answer.
2. For each claim it checks if it is supported, contradicted, or absent in the context.
3. A score in {0, 0.5, 1.0} is assigned per claim; the final score is the mean.

In [ ]:
FAITHFULNESS_SYSTEM = """You are an expert factual auditor.
Given a CONTEXT and a generated ANSWER, evaluate whether the answer is
faithful to the context.

Steps:
1. List each distinct atomic claim in the ANSWER as a JSON array under "claims".
2. For each claim, judge its support: "supported", "contradicted", or "not_in_context".
3. Compute faithfulness_score = (count of supported claims) / (total claims).
   If the answer is "Unanswerable" and the context truly does not support an answer, set score=1.0.
   If the answer is "Unanswerable" but the context does support an answer, set score=0.0.

Respond ONLY with valid JSON matching this schema:
{
  "claims": [{"text": "...", "verdict": "supported|contradicted|not_in_context"}],
  "faithfulness_score": <float 0-1>,
  "reasoning": "<one sentence explanation>"
}"""


def evaluate_faithfulness(context: str, question: str, answer: str,
                           gold: list) -> dict:
    """
    Uses the LLM judge to compute a faithfulness score.

    Returns a dict with keys: claims, faithfulness_score, reasoning.
    Falls back to heuristic score on JSON parse failure.
    """
    # For unanswerable cases, pass the ground truth flag to the judge
    truly_unans_note = (
        '[NOTE: This question is genuinely unanswerable from the context.]'
        if not gold else ''
    )

    prompt = (
        f'{FAITHFULNESS_SYSTEM}\n\n'
        f'CONTEXT:\n{context[:1000]}\n\n'
        f'QUESTION:\n{question}\n\n'
        f'ANSWER:\n{answer}\n\n'
        f'{truly_unans_note}\n'
        f'JSON response:'
    )

    raw = ollama_generate(prompt, model=JUDGE_MODEL, temperature=0.0, max_tokens=512)

    # Extract JSON — handle markdown code fences
    json_match = re.search(r'\{.*\}', raw, re.DOTALL)
    if json_match:
        try:
            parsed = json.loads(json_match.group())
            parsed['faithfulness_score'] = float(parsed.get('faithfulness_score', 0.0))
            parsed['raw_judge_output'] = raw
            return parsed
        except json.JSONDecodeError:
            pass

    # Heuristic fallback — look for a number in the output
    numbers = re.findall(r'\b0?\.?\d+\.?\d*\b', raw)
    score = float(numbers[0]) if numbers else 0.0
    score = min(max(score, 0.0), 1.0)
    return {
        'claims': [],
        'faithfulness_score': score,
        'reasoning': 'JSON parse failed; heuristic score used.',
        'raw_judge_output': raw
    }


print('Faithfulness judge defined.')

## 8. Evaluation Metric 2 — Answer Relevance

### Mathematical Definition

Answer Relevance measures how directly the generated answer **addresses the question**,
independent of the context source:

$$\text{Relevance}(A, Q) = \cos\!\left(\mathbf{e}_A,\, \mathbf{e}_Q\right)$$

where $\mathbf{e}_A$ and $\mathbf{e}_Q$ are sentence embeddings of the answer and question
respectively, computed by a pre-trained bi-encoder (`all-MiniLM-L6-v2`).

**Penalty term**: Answers that are circular (i.e. merely repeat the question) or
excessively verbose receive a length-normalised redundancy penalty:

$$\text{Redundancy}(A, Q) = \frac{|\text{tokens}(A) \cap \text{tokens}(Q)|}{|\text{tokens}(A)| + 1}$$

$$\text{Answer Relevance Score} = \text{Relevance}(A, Q) \times (1 - \lambda \cdot \text{Redundancy}(A, Q))$$

where $\lambda = 0.3$ controls penalty strength.

In [ ]:
# Load a lightweight sentence encoder (downloads ~80 MB once)
print('Loading sentence encoder...')
sent_model = SentenceTransformer('all-MiniLM-L6-v2')
print('Sentence encoder ready.')


REDUNDANCY_LAMBDA = 0.3


def tokenize_simple(text: str) -> set:
    """Lowercase word-level tokens, no stop-words removed."""
    return set(re.findall(r'\b\w+\b', text.lower()))


def answer_relevance_score(question: str, answer: str) -> float:
    """
    Semantic cosine similarity between question and answer embeddings,
    penalised for circular/repetitive answers.

    Returns float in [0, 1].
    """
    if 'unanswerable' in answer.lower().split()[:3]:
        # Abstention answers cannot be semantically relevant to the question
        return 0.0

    q_emb = sent_model.encode(question, convert_to_tensor=True)
    a_emb = sent_model.encode(answer,   convert_to_tensor=True)

    cosine_sim = float(util.cos_sim(q_emb, a_emb))
    cosine_sim = max(0.0, cosine_sim)   # clamp negatives to 0

    # Redundancy penalty
    q_tokens  = tokenize_simple(question)
    a_tokens  = tokenize_simple(answer)
    overlap   = len(q_tokens & a_tokens)
    redundancy = overlap / (len(a_tokens) + 1)

    score = cosine_sim * (1 - REDUNDANCY_LAMBDA * redundancy)
    return round(min(max(score, 0.0), 1.0), 4)


print('Answer relevance scorer defined.')

## 9. Run Full Evaluation

In [ ]:
faithfulness_scores = []
relevance_scores    = []

for r in tqdm(rag_results, desc='LLM Judge evaluation'):
    # --- Faithfulness (LLM-as-a-Judge) ---
    faith_result = evaluate_faithfulness(
        context  = r['context'],
        question = r['question'],
        answer   = r['generated_answer'],
        gold     = r['gold']
    )
    r['faithfulness_result'] = faith_result
    r['faithfulness_score']  = faith_result['faithfulness_score']
    faithfulness_scores.append(r['faithfulness_score'])

    # --- Answer Relevance ---
    rel = answer_relevance_score(r['question'], r['generated_answer'])
    r['relevance_score'] = rel
    relevance_scores.append(rel)

print('Evaluation complete.')

## 10. Aggregate Results & Summary

In [ ]:
mean_faith = np.mean(faithfulness_scores)
mean_rel   = np.mean(relevance_scores)
std_faith  = np.std(faithfulness_scores)
std_rel    = np.std(relevance_scores)

# Separate scores by question type
ans_faith   = [r['faithfulness_score'] for r in rag_results if not r['is_truly_unanswerable']]
unans_faith = [r['faithfulness_score'] for r in rag_results if r['is_truly_unanswerable']]
ans_rel     = [r['relevance_score']    for r in rag_results if not r['is_truly_unanswerable']]
unans_rel   = [r['relevance_score']    for r in rag_results if r['is_truly_unanswerable']]

print('=' * 55)
print(f'{"Metric":<30} {"Mean":>8} {"Std":>8}')
print('=' * 55)
print(f'{"Faithfulness (all)":<30} {mean_faith:>8.4f} {std_faith:>8.4f}')
print(f'{"  → answerable Qs":<30} {np.mean(ans_faith):>8.4f}')
print(f'{"  → unanswerable Qs":<30} {np.mean(unans_faith):>8.4f}')
print(f'{"Answer Relevance (all)":<30} {mean_rel:>8.4f} {std_rel:>8.4f}')
print(f'{"  → answerable Qs":<30} {np.mean(ans_rel):>8.4f}')
print(f'{"  → unanswerable Qs":<30} {np.mean(unans_rel):>8.4f}')
print('=' * 55)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Distribution plots ---
for ax, scores, title, color in [
    (axes[0], faithfulness_scores, 'Faithfulness Score Distribution', 'steelblue'),
    (axes[1], relevance_scores,    'Answer Relevance Score Distribution', 'seagreen')
]:
    ax.hist(scores, bins=20, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(np.mean(scores), color='red', linestyle='--', linewidth=1.8,
               label=f'Mean = {np.mean(scores):.3f}')
    ax.set_xlabel('Score', fontsize=12)
    ax.set_ylabel('Count',  fontsize=12)
    ax.set_title(title,     fontsize=12)
    ax.legend(fontsize=10)
    ax.set_xlim(0, 1.05)

plt.suptitle('Generative QA Evaluation — SQuAD v2.0 (Llama 3 via Ollama)',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('generative_eval_scores.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Scatter: Faithfulness vs. Answer Relevance — coloured by question type
fig, ax = plt.subplots(figsize=(7, 6))

for r in rag_results:
    colour = 'steelblue' if not r['is_truly_unanswerable'] else 'tomato'
    ax.scatter(r['relevance_score'], r['faithfulness_score'],
               c=colour, alpha=0.55, s=45, edgecolors='none')

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='steelblue', label='Answerable'),
    Patch(color='tomato',    label='Unanswerable')
], fontsize=10)

ax.set_xlabel('Answer Relevance', fontsize=12)
ax.set_ylabel('Faithfulness',     fontsize=12)
ax.set_title('Faithfulness vs. Answer Relevance\n(Generative QA, SQuAD v2.0)', fontsize=12)
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-0.05, 1.05)
ax.grid(linestyle=':', alpha=0.5)

plt.tight_layout()
plt.savefig('faithfulness_vs_relevance.png', dpi=150)
plt.show()

## 11. Qualitative Examples — Execution Log

In [ ]:
def show_example(r, label=''):
    print(f'{'='*65}')
    print(f'  {label}')
    print(f'{'='*65}')
    print(f'Question  : {r["question"]}')
    print(f'Context   : {r["context"][:250]}...')
    print(f'Gold      : {r["gold"] if r["gold"] else "<unanswerable>"}')
    print(f'Generated : {r["generated_answer"]}')
    print(f'Faithfulness  : {r["faithfulness_score"]:.4f}')
    print(f'Reasoning     : {r["faithfulness_result"].get("reasoning", "n/a")}')
    print(f'Relevance     : {r["relevance_score"]:.4f}')
    print()


# Best faithful answer (answerable)
best_ans = max(
    (r for r in rag_results if not r['is_truly_unanswerable']),
    key=lambda x: x['faithfulness_score']
)
show_example(best_ans, 'HIGH FAITHFULNESS — Answerable')

# Hallucination example (low faithfulness, answerable)
low_faith = min(
    (r for r in rag_results if not r['is_truly_unanswerable']),
    key=lambda x: x['faithfulness_score']
)
show_example(low_faith, 'LOW FAITHFULNESS — Hallucination Example')

# Correct Unanswerable flag
correct_unans = next(
    (r for r in rag_results if r['is_truly_unanswerable'] and r['predicted_unanswerable']),
    None
)
if correct_unans:
    show_example(correct_unans, 'CORRECT UNANSWERABLE FLAG')

# Missed Unanswerable (false negative)
missed_unans = next(
    (r for r in rag_results if r['is_truly_unanswerable'] and not r['predicted_unanswerable']),
    None
)
if missed_unans:
    show_example(missed_unans, 'MISSED UNANSWERABLE — False Negative')

## 12. Critical Analysis of Evaluation Metrics

### 12.1 Faithfulness — Limitations

**Mathematical bias**: The claim-counting formula
$\text{Faithfulness} = |\text{Supported}| / |\text{All Claims}|$
is unweighted — a minor phrasing claim counts the same as a key factual claim.
A one-sentence answer with one hallucinated claim scores 0.0, whereas a five-sentence
answer with four supported claims and one hallucination scores 0.8.
**Longer answers are therefore systematically rewarded** even if they hallucinate.

**LLM-judge bias**: The judge is the *same model* as the generator (Llama 3),
which introduces self-consistency bias: a model's hallucinations often align with
its own world-knowledge priors, so the judge may not flag what the generator
considers obvious. Cross-model judging (e.g., GPT-4 judging Llama) reduces this.

**Correlation with human judgement**: Studies (RAGAS, 2023; Geval, 2023) show
LLM-as-a-judge faithfulness scores correlate moderately (~0.55–0.75 Pearson)
with human annotations — stronger than ROUGE but weaker than BERTScore.

### 12.2 Answer Relevance — Limitations

**Embedding shortcut**: `all-MiniLM-L6-v2` encodes surface semantics efficiently
but can score a paraphrase of the question (circular answer) as highly relevant
because its vector lies close to the question embedding. The redundancy penalty
partially mitigates this.

**Fluency-vs-substance gap**: A grammatically perfect but vague answer
(e.g., *"The answer depends on the context."*) can score highly because its
embedding aligns with many question types. Neither cosine similarity nor
token overlap detects this kind of semantic evasion.

**Unanswerable handling**: Setting relevance = 0 for abstention outputs is
a design choice — a correct abstention is not *irrelevant*, but measuring its
relevance to the question is ill-defined. A more nuanced treatment would score
the quality of the abstention explanation separately.

**Correlation with human judgement**: Semantic-similarity–based relevance
correlates ~0.60–0.70 with human relevance ratings for factoid QA
(compared to ~0.35–0.45 for BLEU). However, it fails for long-form or
multi-hop answers where the core answer is embedded in a longer passage.

In [ ]:
# Save evaluation results for Part C screenshots / further analysis
output_records = []
for r in rag_results:
    output_records.append({
        'id'                    : r['id'],
        'question'              : r['question'],
        'gold'                  : r['gold'],
        'generated_answer'      : r['generated_answer'],
        'is_truly_unanswerable' : r['is_truly_unanswerable'],
        'predicted_unanswerable': r['predicted_unanswerable'],
        'faithfulness_score'    : r['faithfulness_score'],
        'faithfulness_reasoning': r['faithfulness_result'].get('reasoning', ''),
        'relevance_score'       : r['relevance_score']
    })

with open('rag_evaluation_results.json', 'w') as f:
    json.dump(output_records, f, indent=2)

print(f'Results saved to rag_evaluation_results.json  ({len(output_records)} records)')